In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

def Lag_3(X):
    L_0 = np.exp(-X / 2)
    L_1 = np.exp(-X / 2) * (1 - X)
    L_2 = np.exp(-X / 2) * (1 - 2 * X + X ** 2 / 2)
    return np.column_stack((L_0, L_1, L_2))

def Delta_LSM(eps, S0, K, r, sd, T, I, M, n):
    deltas = []
    for _ in range(n):
        
        rand_nums = np.random.normal(0, np.sqrt(1/M), size=(I, M))
        
    
        price_up = LSM_with_given_rand(S0 + eps, K, r, sd, T, I, M, rand_nums)[0]
        price_down = LSM_with_given_rand(S0 - eps, K, r, sd, T, I, M, rand_nums)[0]
        
        delta = (price_up - price_down) / (2 * eps)
        deltas.append(delta)
    
    avg_delta = np.mean(deltas)
    return round(avg_delta, 4)

def LSM_with_given_rand(S0, K, r, sd, T, I, M, rand_nums):
    dt = 1/M
    M = M * T
    df = np.exp(-r * dt)
    
    S = np.zeros((I, M + 1))
    S[:, 0] = S0
    S[:, 1:] = S0 * np.exp(np.cumsum((r - 0.5 * sd ** 2) * dt + sd * rand_nums, axis=1))
    
    s = S / K
    h = np.maximum(1 - s, 0)
    h[:, 0] = 0
    h_copy = h.copy()

    for i in range(M, 1, -1):
        ITM = h[:, i - 1] > 0
        if np.any(ITM):
            Y = df * h_copy[ITM, i]
            X = s[ITM, i - 1]
            Z = Lag_3(X)
            
            reg = LinearRegression().fit(Z, Y)
            basis = Lag_3(s[:, i - 1])
            cont = reg.predict(basis)
            
            h[:, i - 1] = np.where(h[:, i - 1] <= cont, 0, h[:, i - 1])
            h[h[:, i - 1] > 0, i:] = 0
            h_copy[:, i - 1] = np.where(h[:, i - 1] == 0, h_copy[:, i] * df, h_copy[:, i - 1])
        else:
            h_copy[:, i - 1] = h_copy[:, i] * df

    df_vec = np.power(df, np.arange(M + 1)).reshape(-1, 1)
    disc_CashFlows = h @ df_vec
    price = sum(disc_CashFlows) * K / I
    se = np.sqrt(sum((disc_CashFlows * K - price) ** 2)) / I

    return price


def Gamma_LSM(eps, S0, K, r, sd, T, I, M, n):
    gammas = []
    for _ in range(n):
        
        rand_nums = np.random.normal(0, np.sqrt(1/M), size=(I, M))
        
        
        price_up = LSM_with_given_rand(S0 + eps, K, r, sd, T, I, M, rand_nums)[0]
        price_down = LSM_with_given_rand(S0 - eps, K, r, sd, T, I, M, rand_nums)[0]
        price_mid = LSM_with_given_rand(S0, K, r, sd, T, I, M, rand_nums)[0]
        
        gamma = (price_up - 2 * price_mid + price_down) / (eps ** 2)
        gammas.append(gamma)
    
    avg_gamma = np.mean(gammas)
    return round(avg_gamma, 4)

print(Delta_LSM(0.05,40,40,0.06,0.2,1,10**5,50,1))
